# Prepare baseline ground truth + temporal split

**CPU only** -- plain pandas, no model training, no GPU benefit.

Foundation for every baseline (RBAC, ABAC, ML, DL, RL, graph-structural): attaches ground-truth labels to the combined benchmark and produces a single temporal train/test split that all of them share, so results are directly comparable.

**Now variant-aware.** The combined CSV holds 7 builds x 2 worlds = 14 (variant, world_id) groups (`main`, `sparse_feedback`, `noisy_benign`, `adversarial`, `seed_101`, `seed_103`, `seed_104` -- modelled on the reference ACCESS-IoT dataset's own release-build convention), not just `main`'s original 2 worlds. Labels and the day-cutoff logic need NO changes for this -- ground truth (`gt_cpo_malicious`/`gt_emsp_malicious`/`gt_ev_sybil`) is identical across variants by construction (the variants change observable FEATURES, never the ground truth -- "one factor at a time," same principle ACCESS-IoT's own variants follow), and `day_idx`'s value set is shared real data, so one global day_cutoff naturally stratifies correctly across every variant via the `variant` column carried through in the output. This step just adds a per-variant sanity print of that invariant, so a regression in any one variant's generation would show up here first.

**Why this exists as its own step, not folded into each baseline notebook:** every baseline must be scored against the *same* ground truth and trained/tested on the *same* rows, or the comparison table means nothing. Doing it once here and having every later notebook load the two output CSVs is what makes that guarantee mechanical instead of something to remember to keep in sync by hand.

**Requires:** `v2g_benchmark_multi_world.csv` from `02_generate_v2g_benchmark.ipynb` already in your Drive.

## Step 1: ground-truth labels

Three independent threat axes, each already a real column from the generation pipeline (nothing re-derived from scratch here): `gt_cpo_malicious` (from `authority_malicious`), `gt_emsp_malicious` (from `emsp_id`), `gt_ev_sybil` (from `ev_sybil`). `risk_factor_count`/`composite_risk_score` are evaluation-target design choices, documented in the module's docstring -- not data fields.

In [ ]:
%%writefile benchmark_labels.py
"""
benchmark_labels.py
----------------------
Ground-truth labels shared by EVERY baseline (RBAC, ABAC, ML, DL, RL,
graph-structural) and by TrustMesh itself, so all of them are scored
against the exact same target.

GROUNDING LEDGER:
  - gt_cpo_malicious: directly from injection.py's authority_malicious
    column (1 iff cpo_id == the injected malicious CPO AND day_idx >=
    onset_day). Already a column in v2g_benchmark_multi_world.csv --
    not recomputed here, just carried through.
  - gt_emsp_malicious: ONSET-GATING FIX -- this USED to be a pure, permanent
    function of emsp_id == MALICIOUS_EMSP (identity_assignment.py's eMSP-C),
    with zero temporal variation: the transformer-baseline entity-attention
    proof of concept's own entity-only ablation confirmed this made the axis
    structurally unlearnable from behavior for ANY entity-aggregation model
    (mean- or attention-pooling), since there was never a honest-vs-malicious
    contrast in eMSP-C's own history to learn from, only its name. Now reads
    multi_world_pipeline.py/feature_engineering.py's persisted
    emsp_malicious_eligible column directly (1 iff emsp_id == MALICIOUS_EMSP
    AND day_idx >= emsp_onset_day) -- the same "identity AND onset" shape
    authority_malicious already has for CPO, so the two axes are symmetric.
    A combined CSV that predates this fix won't have that column;
    add_ground_truth_labels raises a clear error rather than silently
    falling back to the old identity-only definition. Independent threat
    axis from CPO billing fraud (independent, staggered onset day).
  - gt_ev_sybil: directly from enriched_features.py's ev_sybil column.
    Independent threat axis from both of the above.
  - risk_factor_count / composite_risk_score: NOT data fields -- these
    are EVALUATION-TARGET design choices made HERE, not anything the
    generation pipeline produces or implies. Said plainly so they can
    be cited precisely in the paper rather than mistaken for something
    collected "as-is": risk_factor_count is a simple count (0-3) of how
    many of the three independent threat axes are active for a given
    session; composite_risk_score is that count turned into [0,1] under
    an EQUAL-WEIGHTING assumption across axes (AXIS_WEIGHTS below) --
    not a claim that CPO billing fraud, eMSP credential issues, and EV
    Sybil attacks carry equal real-world severity.

WHY NOT A BAKED-IN "DECISION" COLUMN HERE: every baseline's own
continuous score gets passed through decision_engine.trust_weighted_decision()
at evaluation time (04_performance_analysis.ipynb), not pre-computed here.
That keeps the PERMIT / CONDITIONAL-PERMIT / DENY granularity a property
of each model's score, not of the ground truth -- which is what lets the
benchmark show every baseline (not just TrustMesh) supporting that same
granularity, and lets a user push to a finer granularity later just by
changing threshold cut points, with nothing in this module to revisit.
"""

import numpy as np
import pandas as pd

MALICIOUS_EMSP = "eMSP-C"  # mirrors identity_assignment.py; duplicated here
                            # deliberately so this module has no import-time
                            # dependency on trustmesh_pkg's package layout

AXIS_WEIGHTS = {"cpo": 1.0, "emsp": 1.0, "ev": 1.0}  # documented equal-weighting
                                                        # assumption, see module docstring


def add_ground_truth_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["gt_cpo_malicious"] = df["authority_malicious"].astype(int)

    if "emsp_malicious_eligible" not in df.columns:
        raise KeyError(
            "Column 'emsp_malicious_eligible' not found. This combined CSV predates "
            "the eMSP onset-gating fix (see multi_world_pipeline.py / "
            "feature_engineering.py, and this module's GROUNDING LEDGER above) -- "
            "regenerate the dataset (02_generate_v2g_benchmark.ipynb) before running "
            "this baseline. Refusing to silently fall back to the old, structurally-"
            "unlearnable identity-only definition (emsp_id == MALICIOUS_EMSP)."
        )
    df["gt_emsp_malicious"] = df["emsp_malicious_eligible"].astype(int)
    df["gt_ev_sybil"] = df["ev_sybil"].astype(int)

    df["risk_factor_count"] = (
        df["gt_cpo_malicious"] + df["gt_emsp_malicious"] + df["gt_ev_sybil"]
    )
    total_weight = sum(AXIS_WEIGHTS.values())
    df["composite_risk_score"] = (
        AXIS_WEIGHTS["cpo"] * df["gt_cpo_malicious"]
        + AXIS_WEIGHTS["emsp"] * df["gt_emsp_malicious"]
        + AXIS_WEIGHTS["ev"] * df["gt_ev_sybil"]
    ) / total_weight
    return df


def summarize_labels(df: pd.DataFrame) -> None:
    print("risk_factor_count distribution (0 = clean, 3 = all three threat axes active):")
    print(df["risk_factor_count"].value_counts().sort_index())
    print(f"\nAny risk factor present: {(df['risk_factor_count'] > 0).mean():.4f}")
    for axis in ["gt_cpo_malicious", "gt_emsp_malicious", "gt_ev_sybil"]:
        print(f"  {axis} rate: {df[axis].mean():.4f}")
    print(f"\ncomposite_risk_score describe:\n{df['composite_risk_score'].describe()}")


if __name__ == "__main__":
    import os
    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = add_ground_truth_labels(df)
    summarize_labels(df)


## Step 2: temporal train/test split

Split by `day_idx`, not randomly and not by `world_id`/`variant` -- see `temporal_split.py`'s docstring for why. The same day cutoff is used across every variant and world, since `day_idx` is real data shared across all of them.

In [ ]:
%%writefile temporal_split.py
"""
temporal_split.py
--------------------
Train/test split for baseline evaluation: temporal within each world,
not random and not by-world.

WHY TEMPORAL: day_idx is REAL (derived from actual ACN-Data connection
timestamps, see raw_preprocess.py) and IDENTICAL in range across both
worlds -- only the synthetic identity/fraud fields differ by world, not
the real session timing. injection.py's attacks are keyed on day_idx
(onset_day), so a model that has only ever seen days before the cutoff
has a genuinely harder, more realistic job on the test split than a
model trained on a random shuffle that leaks future-day sessions into
training. This is the right split for a benchmark meant to measure "how
well/fast does each baseline detect or adapt to a trust problem," not
just "can it memorize this distribution."

Applied with the SAME day cutoff across all worlds (computed once, on
the full combined frame), since day_idx's distribution is shared real
data, not an independent per-world draw.

ONE DISCLOSED CAVEAT: enriched_features.add_baseline_charge_duration's
station+hour empirical-median fallback was computed over the WHOLE
dataset (train+test combined) when the benchmark CSV was generated, not
separately per split -- a minor leakage source for that one contextual
feature column specifically. Harmless for anything that doesn't use
baseline_charge_duration_h as a predictor of risk (it isn't one), but
worth stating plainly rather than silently ignoring, consistent with
this project's disclosure discipline.
"""

import numpy as np
import pandas as pd

TRAIN_FRACTION_OF_DAYS = 0.7  # documented split point, not tuned


def compute_day_cutoff(df: pd.DataFrame, train_fraction: float = TRAIN_FRACTION_OF_DAYS) -> int:
    days = np.sort(df["day_idx"].unique())
    cutoff_idx = int(len(days) * train_fraction)
    cutoff_idx = min(max(cutoff_idx, 1), len(days) - 1)  # keep both splits non-empty
    return int(days[cutoff_idx])


def temporal_train_test_split(df: pd.DataFrame, day_cutoff: int = None):
    if day_cutoff is None:
        day_cutoff = compute_day_cutoff(df)
    train = df[df["day_idx"] < day_cutoff].copy()
    test = df[df["day_idx"] >= day_cutoff].copy()
    print(f"[split] day_cutoff={day_cutoff}  train={len(train)} rows "
          f"(days {train['day_idx'].min()}-{train['day_idx'].max()})  "
          f"test={len(test)} rows (days {test['day_idx'].min()}-{test['day_idx'].max()})")
    for name, part in [("train", train), ("test", test)]:
        rate = (part["risk_factor_count"] > 0).mean() if "risk_factor_count" in part.columns else float("nan")
        print(f"  {name}: any-risk-factor rate = {rate:.4f}")
    return train, test, day_cutoff


if __name__ == "__main__":
    import os
    import benchmark_labels

    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = benchmark_labels.add_ground_truth_labels(df)
    train, test, cutoff = temporal_train_test_split(df)
    print(f"\nWorld breakdown -- train:\n{train['world_id'].value_counts()}")
    print(f"World breakdown -- test:\n{test['world_id'].value_counts()}")


In [ ]:
import os
import pandas as pd
import benchmark_labels
import temporal_split

COMBINED_CSV = os.environ.get('V2G_COMBINED_CSV', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv')
OUT_DIR = os.environ.get('V2G_LABELS_SPLITS_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds')

df = pd.read_csv(COMBINED_CSV)
df = benchmark_labels.add_ground_truth_labels(df)
benchmark_labels.summarize_labels(df)

if 'variant' in df.columns:
    print(f'\nVariants present: {sorted(df["variant"].unique())}')
    print('\nRows per variant:')
    print(df['variant'].value_counts().sort_index())
    print('\nGround-truth rate per variant (expect ~identical across all 7 -- '
          'variants change OBSERVABLE features, never ground truth):')
    print(df.groupby('variant')[['gt_cpo_malicious', 'gt_emsp_malicious', 'gt_ev_sybil']]
          .mean().round(4))

train, test, day_cutoff = temporal_split.temporal_train_test_split(df)

if 'variant' in df.columns:
    print('\nTrain rows per variant:')
    print(train['variant'].value_counts().sort_index())
    print('\nTest rows per variant:')
    print(test['variant'].value_counts().sort_index())

train_path = os.path.join(OUT_DIR, 'v2g_benchmark_train.csv')
test_path = os.path.join(OUT_DIR, 'v2g_benchmark_test.csv')
train.to_csv(train_path, index=False)
test.to_csv(test_path, index=False)
print(f'\nSaved train -> {train_path}  ({len(train)} rows)')
print(f'Saved test  -> {test_path}  ({len(test)} rows)')
os.environ['V2G_TRAIN_CSV'] = train_path
os.environ['V2G_TEST_CSV'] = test_path

## Done

`v2g_benchmark_train.csv` / `v2g_benchmark_test.csv` in the same `worlds/` folder -- every baseline notebook from here on loads these two files directly rather than re-deriving labels or re-splitting. Paste back the printed label distribution, the per-variant ground-truth table, and split sizes.